# レッスン05 - エージェンティックRAG


## セットアップ

このノートブックは、Microsoft Agent Framework を使用したエージェンティック RAG（Retrieval-Augmented Generation）パターンを示します。

**前提条件:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search サービスのエンドポイント
- `AZURE_SEARCH_API_KEY` — Azure AI Search の API キー
- 環境変数で設定された Azure OpenAI デプロイメント
- Azure CLI による認証済み（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAGとは何ですか？

従来のRAGは固定されたパイプラインに従います：文書を取得し、その後に応答を生成します。<strong>Agentic RAG</strong>はより進んで、情報を取得する<strong>タイミング</strong>や<strong>方法</strong>をエージェントに自律的に決定させます。

Agentic RAGでは、エージェントは以下を行うことができます：
- 質問に答える前に取得が必要かどうかを<strong>決定</strong>する
- どのデータソースやツールを問い合わせるかを<strong>選択</strong>する
- 取得した結果を<strong>評価</strong>し、最初の試みが不十分な場合は追加の取得を行う
- 複数の取得ステップからの情報を組み合わせて一貫した回答を<strong>生成</strong>する

これにより、静的な取得してから生成するパイプラインと比べて、エージェントはより柔軟かつ正確になります。


## 検索ツールの作成

Agentic RAGでは、外部データソースがエージェントが必要に応じて呼び出せる<strong>ツール</strong>としてラップされています。これにより、エージェントは検索を必須のステップではなく、実行できるアクションの一つとして扱うことができます。

以下では、旅行に関する知識ベースを定義し、目的地情報を調べるためにエージェントが呼び出せるツールとして公開します。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAGエージェントの構築

ここでは、<strong>必ず回答する前に情報を取得するよう指示された</strong>エージェントを作成します。このエージェントは、`search_travel_knowledge` ツールを使用して、自分の学習データに頼るのではなく、ナレッジベースに基づいて回答を行います。


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 反復検索 — メイカーチェッカーパターン

Agentic RAG の大きな利点は<strong>反復検索</strong>です。エージェントは複数回の検索を行い、最初の調査結果を検証、修正、または拡張できます — これは「メイカーチェッカー」ワークフローに似ています：

1. <strong>メイカー段階</strong>：エージェントが初期情報を取得し、回答案を作成します。
2. <strong>チェッカー段階</strong>：エージェントが追加の検索を行い、詳細を検証したり、欠落部分を埋めたりします。

以下では、複数の目的地を比較する必要がある質問がエージェントに投げかけられ、複数回の検索を促しています。


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

このレッスンでは、Microsoft Agent Framework を使って<strong>Agentic RAG</strong>システムを構築する方法を学びました。

- <strong>Agentic RAG</strong>はエージェントが情報をいつ取得するかを自律的に判断できるため、取得が固定的ではなく動的になります。
- <strong>ツールとしてのデータソース</strong>：外部の知識ベース（Azure AI Search など）は、エージェントが呼び出せるツールとしてラップされます。
- <strong>反復的な取得</strong>：メーカー・チェッカーのパターンにより、エージェントは最終回答を作成する前に複数回の取得（検索、検証、洗練）を行うことができます。

実際の運用では、大規模な旅行文書の取得を扱うために、メモリ内の `TRAVEL_KNOWLEDGE_BASE` を実際の Azure AI Search インデックスに置き換えます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
